In [5]:
!git clone -b dev https://github.com/harishkulkarni10/ai-research-assistant-rag.git

fatal: destination path 'ai-research-assistant-rag' already exists and is not an empty directory.


# Dense embeddings for RAG
- We convert text chunks into dense vector embeddings

In [6]:
# Load the chunks
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path("/content/ai-research-assistant-rag")
CHUNKS_PATH = PROJECT_ROOT / "data" / "processed" / "chunks.parquet"

chunks_df = pd.read_parquet(CHUNKS_PATH)

print(f"Loaded {len(chunks_df):,} chunks")
chunks_df.head(2)

Loaded 12,717 chunks


,chunk_id,paper_id,text,section,source,token_count,position_in_paper
0,paper_0_chunk_0,0,the observational result that high - redshift ...,None,arxiv,978,0
1,paper_0_chunk_1,0,we use the absorption system parameters derive...,None,arxiv,886,1


In [8]:
# Installing embedding dependencies

!pip install -U sentence-transformers accelerate torch --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 838.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 94.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 65.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 74.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.3/39.3 MB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.0/90.0 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.

# BGE embeddings
- Embed each chunk's text and store the resulting vectors alongside existing metadata


In [9]:
# Load the BGE model

import torch
from sentence_transformers import SentenceTransformer, util

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('using device: ', device)

bge_model = SentenceTransformer(
    'BAAI/bge-base-en-v1.5',
    device=device
)

using device:  cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [11]:
# Embedding chunks

texts = chunks_df['text'].tolist()

bge_embeddings = bge_model.encode(
    texts,
    batch_size = 32,
    show_progress_bar = True,
    normalize_embeddings=True
)

print(f"Shape of the embeddings: {bge_embeddings.shape}")

Batches:   0%|          | 0/398 [00:00<?, ?it/s]

Shape of the embeddings: (12717, 768)


In [12]:
chunks_bge = chunks_df.copy()
chunks_bge['embedding'] = bge_embeddings.tolist()

output_path = PROJECT_ROOT / "data" / "processed" / "chunks_bge.parquet"
chunks_bge.to_parquet(output_path, index=False)

print(f"BGE embeddings saved to {output_path}")
chunks_bge.head(2)

BGE embeddings saved to /content/ai-research-assistant-rag/data/processed/chunks_bge.parquet


,chunk_id,paper_id,text,section,source,token_count,position_in_paper,embedding
0,paper_0_chunk_0,0,the observational result that high - redshift ...,None,arxiv,978,0,"[0.031177345663309097, 0.012894019484519958, 0..."
1,paper_0_chunk_1,0,we use the absorption system parameters derive...,None,arxiv,886,1,"[0.03079364262521267, 0.038800131529569626, 0...."


# E5

In [13]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("/content/ai-research-assistant-rag")
CHUNKS_PATH = PROJECT_ROOT / "data" / "processed" / "chunks.parquet"

chunks_df = pd.read_parquet(CHUNKS_PATH)

print(f"Loaded {len(chunks_df):,} chunks")
chunks_df.head(2)

Loaded 12,717 chunks


,chunk_id,paper_id,text,section,source,token_count,position_in_paper
0,paper_0_chunk_0,0,the observational result that high - redshift ...,None,arxiv,978,0
1,paper_0_chunk_1,0,we use the absorption system parameters derive...,None,arxiv,886,1


In [14]:
# Load E5

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('using device: ', device)

e5_model = SentenceTransformer(
    'intfloat/e5-base-v2',
    device=device
)

using device:  cuda


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/650 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/314 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

In [15]:
e5_texts = ['passage: ' + t for t in chunks_df['text'].tolist()]

print("Simple E5 input: ")
print(e5_texts[0][:200])

Simple E5 input: 
passage: the observational result that high - redshift ly @xmath0  absorption systems appear not to cluster strongly in redshift ( e.g. sargent 1980 ) has driven most discussion about the origin of th


In [17]:
# Generate E5 embeddings

e5_embeddings = e5_model.encode(
    e5_texts,
    batch_size = 32,
    show_progress_bar = True,
    normalize_embeddings=True
)

print(f"E5 embeddings shape: {e5_embeddings.shape}")

Batches:   0%|          | 0/398 [00:00<?, ?it/s]

E5 embeddings shape: (12717, 768)


In [18]:
chunks_e5 = chunks_df.copy()
chunks_e5["embedding"] = list(e5_embeddings)

output_path = PROJECT_ROOT / "data" / "processed" / "chunks_e5.parquet"
chunks_e5.to_parquet(output_path, index=False)

print(f"E5 embeddings saved to {output_path}")

E5 embeddings saved to /content/ai-research-assistant-rag/data/processed/chunks_e5.parquet


In [19]:
# Reload and verify
verify_df = pd.read_parquet(output_path)

print(f"Reloaded {len(verify_df):,} chunks")
print("Embedding dimension:", len(verify_df.iloc[0]["embedding"]))

Reloaded 12,717 chunks
Embedding dimension: 768


# Let's compare these 2 embeddings as a sample for now - and then decide which embeddings to move forward with in the retrieval phase

In [23]:
import numpy as np
from sentence_transformers.util import cos_sim

chunks_bge = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "chunks_bge.parquet")
bge_emb = np.array(chunks_bge['embedding'].tolist(), dtype=np.float32)

chunks_e5 = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "chunks_e5.parquet")
e5_emb = np.array(chunks_e5['embedding'].tolist(), dtype=np.float32)

# Sample queries
sample_queries = [
    "What are the key innovations in Transformer architecture?",
    "Explain self-attention mechanism in detail",
    "Recent advances in diffusion models",
    "How does RAG reduce hallucinations?",
    "Comparison between BERT and GPT models"
]

print("Sample Retrieval Comparison (Top-1 chunk preview)")

for query in sample_queries:
    print(f"\nQuery: {query}")

    # Embed query with both models
    q_bge = bge_model.encode(query, normalize_embeddings=True)
    q_e5 = e5_model.encode("query: " + query, normalize_embeddings=True)  # E5 uses prefix

    # Find top-1 for BGE
    sim_bge = cos_sim(q_bge, bge_emb)[0]
    top_idx_bge = np.argmax(sim_bge).item() # Extract integer value
    print(f"BGE Top-1 score: {sim_bge[top_idx_bge]:.4f}")
    print("BGE Top chunk preview:", chunks_bge.iloc[top_idx_bge]['text'][:300] + "...")

    # Find top-1 for E5
    sim_e5 = cos_sim(q_e5, e5_emb)[0]
    top_idx_e5 = np.argmax(sim_e5).item() # Extract integer value
    print(f"E5 Top-1 score: {sim_e5[top_idx_e5]:.4f}")
    print("E5 Top chunk preview:", chunks_e5.iloc[top_idx_e5]['text'][:300] + "...")

Sample Retrieval Comparison (Top-1 chunk preview)

Query: What are the key innovations in Transformer architecture?
BGE Top-1 score: 0.6305
BGE Top chunk preview: furthermore , we describe the different proposed ntl detection models and explain why they are particularly scalable to big data sets . section [ chapter : eval ] presents experimental results and a comparison of the models on the data for different ntl proportions in the data . 
 section [ chapter ...
E5 Top-1 score: 0.7947
E5 Top chunk preview: it is worth mentioning that due to insufficient margin of short - circuit capacity in some mv grids , nowadays the dispersed power plants have to be directly connected to the hv grids through expensive generator transformers . 
 however , this considerable investment could be avoided by installing t...

Query: Explain self-attention mechanism in detail
BGE Top-1 score: 0.6608
BGE Top chunk preview: sanjuan , and k. aihara , chaos * 16 * , 013113 ( 2006 ) . 
 x. shi and q. lu , physic